### Система контроля техники безопасности (PPE Detection)

Задача: Детекция касок и жилетов на строительной площадке.

Стек:

 - ML: YOLOv8 (обучение на открытом датасете Hard Hat Workers), OpenCV (рисование рамок), ONNX (инференс).
 - Backend: FastAPI (эндпоинт для загрузки видео/картинок), Redis (очередь задач, если видео длинное).
 - DB: PostgreSQL (сохранение логов: "время, камера, нарушение: нет каски").
 - DevOps: Docker Compose, Nginx.

In [12]:
!pip install -U ultralytics

In [13]:
import kagglehub
path = kagglehub.dataset_download("muhammetzahitaydn/hardhat-vest-dataset-v3")

Using Colab cache for faster access to the 'hardhat-vest-dataset-v3' dataset.


In [14]:
import yaml
import os

dataset_root = path

data_config = {
    'path': dataset_root,
    'train': 'images/train',   # Относительный путь
    'val': 'images/val',       # Относительный путь
    'test': 'images/test',     # Относительный путь
    'nc': 4,
    'names': ['helmet', 'vest', 'head', 'person']
}

yaml_path = 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(data='data.yaml', epochs=30, imgsz=640)
path_to_best_pt = 'runs/detect/train/weights/best.pt'
best_model = YOLO(path_to_best_pt)
new_model_path = best_model.export(format="tflite", int8=True, data="data.yaml")

print(f"Модель успешно экспортирована в: {new_model_path}")

Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True,

In [9]:
from ultralytics import YOLO

model = YOLO('runs/detect/train8/weights/best.pt')
test_path = '/root/.cache/kagglehub/datasets/muhammetzahitaydn/hardhat-vest-dataset-v3/versions/2/images/test'
results = model.predict(source=test_path, save=True)

print(results)

FileNotFoundError: [Errno 2] No such file or directory: 'runs/detect/train8/weights/best.pt'

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO('runs/detect/train8/weights/best.pt')

image_path = '/root/.cache/kagglehub/datasets/muhammetzahitaydn/hardhat-vest-dataset-v3/versions/2/images/test/000010.jpg'
img = cv2.imread(image_path)

if img is None:
    print(f"Ошибка: Не удалось загрузить изображение по пути {image_path}")
    exit()

results = model(img, verbose=False)

colors = {
    0: (255, 0, 0),     # Синий для Helmet
    1: (0, 255, 255),   # Желтый для Vest
    2: (0, 255, 0),     # Зеленый для Head
    3: (0, 0, 255)      # Красный для Person
}

for result in results:
    boxes = result.boxes

    for box in boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        class_name = model.names[cls_id]
        label = f"{class_name} {conf:.2f}"
        color = colors.get(cls_id, (255, 255, 255))
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        (text_w, text_h), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
        cv2.rectangle(img, (x1, y1 - text_h - 10), (x1 + text_w, y1), color, -1)
        cv2.putText(img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)

cv2.imwrite('result_cv2.jpg', img)

True